# DS Workbench: data-science work as `ds-user-1`

Everything in this notebook runs **as `ds-user-1`**, a member of the `data-science` IAM group, not as the admin. So you work with exactly the permissions a data scientist at the bank would have:

| May | May not |
|---|---|
| Read `bank-lake/loans/` and query the `loans` table with Athena | Delete source data |
| Write anywhere under `bank-lake/sandbox/data-science/` | Read `bank-lake/pii/` (personal data) |
| Register SageMaker models and start training jobs | Deploy endpoints (that's MLOps' job) |

**Before you start:** the emulator must be running, and `labs/aws-basics/data_science_group.py` must have run once since it started. That script creates the group, the user and the loan data.

## 1. Connect to the emulator

These settings suit an emulator running in Docker on your PC. The loop finds `emu.py` whether this notebook sits next to the `claude-aws-emulation` folder or inside it.

In [1]:
import os, sys, time
os.environ.setdefault("EMULATOR_ENDPOINT", "https://127.0.0.1:9297")   # where the emulator listens
os.environ.setdefault("EMULATOR_INSECURE", "1")                        # accept its self-signed certificate
for folder in ["claude-aws-emulation", ".", "..", "../.."]:
    if os.path.exists(os.path.join(folder, "emu.py")):
        sys.path.insert(0, os.path.abspath(folder))
        break
import urllib3; urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)
import emu
print("Emulator healthy:", emu.health())

Emulator healthy: True


## 2. Sign in as `ds-user-1`

The admin client is used for one thing only: issuing ds-user-1 an access key. Every client after that signs with ds-user-1's own key.

In [2]:
iam = emu.client("iam")                                    # admin, used only to issue the key
keys = iam.list_access_keys(UserName="ds-user-1")["AccessKeyMetadata"]
if len(keys) >= 2:                                         # AWS allows 2 keys per user: retire the oldest
    oldest = min(keys, key=lambda k: k["CreateDate"])
    iam.delete_access_key(UserName="ds-user-1", AccessKeyId=oldest["AccessKeyId"])
ds_key = iam.create_access_key(UserName="ds-user-1")["AccessKey"]
time.sleep(1)                                              # new keys take a moment, as on AWS

s3     = emu.client("s3",     credentials=ds_key, who="ds-user-1")
athena = emu.client("athena", credentials=ds_key, who="ds-user-1")
glue   = emu.client("glue",   credentials=ds_key, who="ds-user-1")
print("Working as:", emu.client("sts", credentials=ds_key).get_caller_identity()["Arn"])

Working as: arn:aws:iam::000000000000:user/ds-user-1


## 3. Helpers for data work

`read_csv` loads a file from S3 into pandas. `run_sql` runs a query with Athena, waits for it, collects every page of results and turns numbers back into numbers.

In [3]:
import pandas as pd

def read_csv(key, bucket="bank-lake"):
    """Read a CSV from S3 straight into pandas."""
    return pd.read_csv(s3.get_object(Bucket=bucket, Key=key)["Body"])

def run_sql(sql, database="lending"):
    """Run SQL with Athena and return a DataFrame."""
    qid = athena.start_query_execution(QueryString=sql, QueryExecutionContext={"Database": database},
              ResultConfiguration={"OutputLocation": "s3://bank-athena-results/"})["QueryExecutionId"]
    while True:                                            # wait for the query to finish
        status = athena.get_query_execution(QueryExecutionId=qid)["QueryExecution"]["Status"]
        if status["State"] in ("SUCCEEDED", "FAILED", "CANCELLED"):
            break
        time.sleep(0.5)
    if status["State"] != "SUCCEEDED":
        raise RuntimeError(f'{status["State"]}: {status.get("StateChangeReason")}')
    rows, columns, token = [], None, None
    while True:                                            # results arrive 1,000 rows per page
        page = athena.get_query_results(QueryExecutionId=qid, **({"NextToken": token} if token else {}))
        columns = columns or [c["Name"] for c in page["ResultSet"]["ResultSetMetadata"]["ColumnInfo"]]
        rows += [[d.get("VarCharValue") for d in r["Data"]] for r in page["ResultSet"]["Rows"]]
        token = page.get("NextToken")
        if not token:
            break
    if rows and rows[0] == columns:                        # the first row repeats the headers
        rows = rows[1:]
    df = pd.DataFrame(rows, columns=columns)
    for col in df.columns:                                 # Athena returns everything as text
        try:
            df[col] = pd.to_numeric(df[col])
        except (ValueError, TypeError):
            pass
    return df

## 4. Explore the data

What's in the bucket, which tables Glue knows, and a first Athena query.

In [4]:
for o in s3.list_objects_v2(Bucket="bank-lake")["Contents"]:
    print(o["Key"], o["Size"])
print("Tables:", [t["Name"] for t in glue.get_tables(DatabaseName="lending")["TableList"]])
loans = read_csv("loans/loans.csv")
print(loans.shape)
loans.head()

loans/loans.csv 131383
models/ds-experiment-1795000505334/output/model.tar.gz 435
models/ds-train-1795000520/output/model.tar.gz 437
models/ds-train-1795000619/output/model.tar.gz 438
pii/customers.csv 32
sandbox/data-science/models/default-model-v1/model.tar.gz 716
sandbox/data-science/notes.txt 13
sandbox/data-science/scores/loan_scores.csv 47879
sandbox/data-science/segment_summary.csv 99
Tables: ['loans']
(2000, 11)


,loan_id,segment,product,region,credit_score,annual_income,loan_amount,dti,tenure_months,days_past_due,defaulted
0,LN100000,Self-employed,Personal loan,West,622,47400,25000,0.261,84,30,0
1,LN100001,SME,Auto loan,South,564,28800,166000,0.653,48,60,1
2,LN100002,SME,Home loan,South,718,45400,71000,0.454,84,0,0
3,LN100003,Salaried,Credit card,South,610,63400,345000,0.359,84,90,1
4,LN100004,Salaried,Auto loan,West,659,53900,40000,0.413,48,30,1


In [5]:
summary = run_sql("""SELECT segment, COUNT(*) AS loans,
                            ROUND(AVG(CAST(defaulted AS DOUBLE)), 3) AS default_rate
                     FROM loans GROUP BY segment ORDER BY default_rate DESC""")
s3.put_object(Bucket="bank-lake", Key="sandbox/data-science/segment_summary.csv", Body=summary.to_csv(index=False).encode())
summary

,segment,loans,default_rate
0,Self-employed,478,0.272
1,Salaried,1013,0.240
2,Agri,193,0.212
3,SME,316,0.209


## 5. Connect to SageMaker

`ROLE` is the execution role: SageMaker runs jobs as this role, not as you. The team's policy lets it pass this one role and no other.

In [6]:
from botocore.exceptions import ClientError
sm = emu.client("sagemaker", credentials=ds_key, who="ds-user-1")
ACCOUNT = emu.client("sts", credentials=ds_key).get_caller_identity()["Account"]
ROLE = f"arn:aws:iam::{ACCOUNT}:role/ds-sagemaker-execution"
IMAGE = "683313688378.dkr.ecr.us-east-1.amazonaws.com/sagemaker-scikit-learn:1.2-1-cpu-py3"
print("Training jobs so far:", [j["TrainingJobName"] for j in sm.list_training_jobs()["TrainingJobSummaries"]])

Training jobs so far: ['ds-experiment-1795000505334', 'ds-train-1795000520', 'ds-train-1795000619']


## 6. Train a credit-default model here, and save it to S3

The emulator runs no training compute, so the real model is trained in this notebook. It's saved as `model.tar.gz`, the package format SageMaker expects.

In [7]:
import io, tarfile, pickle
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

FEATURES = ["credit_score", "annual_income", "loan_amount", "dti", "tenure_months", "days_past_due"]
X_tr, X_te, y_tr, y_te = train_test_split(loans[FEATURES], loans["defaulted"], test_size=0.25,
                                          random_state=7, stratify=loans["defaulted"])
model = LogisticRegression(max_iter=1000).fit(X_tr, y_tr)
print("Hold-out AUC:", round(roc_auc_score(y_te, model.predict_proba(X_te)[:, 1]), 3))

buf = io.BytesIO()
with tarfile.open(fileobj=buf, mode="w:gz") as tar:
    blob = pickle.dumps(model); info = tarfile.TarInfo("model.pkl"); info.size = len(blob)
    tar.addfile(info, io.BytesIO(blob))
ART = "sandbox/data-science/models/default-model-v1/model.tar.gz"
s3.put_object(Bucket="bank-lake", Key=ART, Body=buf.getvalue())
print("Saved to s3://bank-lake/" + ART)

Hold-out AUC: 0.789
Saved to s3://bank-lake/sandbox/data-science/models/default-model-v1/model.tar.gz


## 7. Register the model in SageMaker

A SageMaker model is a record: which container image to run, where the artifact is, and which role to run it as.

In [8]:
MODEL_NAME = f"ds-default-model-{int(time.time())}"
sm.create_model(ModelName=MODEL_NAME, ExecutionRoleArn=ROLE,
                PrimaryContainer={"Image": IMAGE, "ModelDataUrl": f"s3://bank-lake/{ART}"})
d = sm.describe_model(ModelName=MODEL_NAME)
print(d["ModelName"], "->", d["PrimaryContainer"]["ModelDataUrl"])

ds-default-model-1795000636 -> s3://bank-lake/sandbox/data-science/models/default-model-v1/model.tar.gz


## 8. Start a training job and watch its status

The job goes through SageMaker's real stages, but the emulator trains nothing: the output is a placeholder with an empty model. Use this to learn the job API and status handling.

In [9]:
JOB = f"ds-train-{int(time.time())}"
sm.create_training_job(TrainingJobName=JOB, RoleArn=ROLE,
    AlgorithmSpecification={"TrainingImage": IMAGE, "TrainingInputMode": "File"},
    HyperParameters={"max_iter": "1000"},
    InputDataConfig=[{"ChannelName": "train", "DataSource": {"S3DataSource": {
        "S3DataType": "S3Prefix", "S3Uri": "s3://bank-lake/loans/", "S3DataDistributionType": "FullyReplicated"}}}],
    OutputDataConfig={"S3OutputPath": "s3://bank-lake/models/"},
    ResourceConfig={"InstanceType": "ml.m5.large", "InstanceCount": 1, "VolumeSizeInGB": 5},
    StoppingCondition={"MaxRuntimeInSeconds": 600})
while True:
    d = sm.describe_training_job(TrainingJobName=JOB)
    print(d["TrainingJobStatus"], "-", d.get("SecondaryStatus"))
    if d["TrainingJobStatus"] in ("Completed", "Failed", "Stopped"):
        break
    time.sleep(5)
print("Output:", d["ModelArtifacts"]["S3ModelArtifacts"])

InProgress - Starting


InProgress - Training


Completed - Completed
Output: s3://bank-lake/models/ds-train-1795000636/output/model.tar.gz


## 9. Batch scoring: load the model back from S3 and score loans

This is what a nightly scoring job does: fetch the approved model, score the book, save the scores. Endpoints can't run in the emulator, so scoring happens here.

In [10]:
body = s3.get_object(Bucket="bank-lake", Key=ART)["Body"].read()
with tarfile.open(fileobj=io.BytesIO(body)) as tar:
    loaded = pickle.loads(tar.extractfile("model.pkl").read())

scores = loans[["loan_id", "segment"]].copy()
scores["probability_of_default"] = loaded.predict_proba(loans[FEATURES])[:, 1].round(3)
s3.put_object(Bucket="bank-lake", Key="sandbox/data-science/scores/loan_scores.csv", Body=scores.to_csv(index=False).encode())
print("Scored", len(scores), "loans; saved to sandbox/data-science/scores/loan_scores.csv")
scores.sort_values("probability_of_default", ascending=False).head()

Scored 2000 loans; saved to sandbox/data-science/scores/loan_scores.csv


,loan_id,segment,probability_of_default
1321,LN101321,Salaried,0.971
681,LN100681,Self-employed,0.937
543,LN100543,SME,0.923
1585,LN101585,Salaried,0.919
683,LN100683,SME,0.916


## 10. Check the guard rails

These should be refused with AccessDenied. If your emulator runs with `--auth off`, they may be allowed instead: that mode doesn't check permissions.

In [11]:
def attempt(label, call):
    try:
        call(); print(f"{label}: allowed")
    except ClientError as e:
        print(f"{label}: {e.response['Error']['Code']}")

attempt("Write outside the sandbox", lambda: s3.put_object(Bucket="bank-lake", Key="loans/edited.csv", Body=b"x"))
attempt("Delete source data",       lambda: s3.delete_object(Bucket="bank-lake", Key="loans/loans.csv"))
attempt("Read personal data",       lambda: s3.get_object(Bucket="bank-lake", Key="pii/customers.csv"))

Write outside the sandbox: AccessDenied
Delete source data: AccessDenied
Read personal data: AccessDenied
